In [1]:
# -- MLOps bootstrap (kept compatible with notebook execution) --
import os
import warnings
from pathlib import Path

import mlflow
import numpy as np
import random

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

try:
    import torch
    torch.manual_seed(SEED)
except ImportError:
    pass

try:
    import tensorflow as tf
    tf.random.set_seed(SEED)
except ImportError:
    pass

mlflow.set_tracking_uri("sqlite:///mlflow.db")
_exp = mlflow.set_experiment("54_Nuanced_Translation_Tuner")
print(f"MLflow experiment: {_exp.name}")

e:\Github\Machine-Learning-Projects\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.7) doesn't match a supported version!
  warnings.warn(
e:\Github\Machine-Learning-Projects\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


MLflow experiment: 54_Nuanced_Translation_Tuner


# 54. Nuanced Translation Tuner

This notebook demonstrates a practical workflow for tuning **domain-specific translation style** using a literary domain example from English to French.

## What You Will Learn
- How to curate a domain-specific parallel dataset from a public source.
- How to define style-aware evaluation beyond plain lexical overlap.
- How to run a fast baseline now and optionally run full fine-tuning later.

## Dataset Source
- Hugging Face `opus_books` dataset (parallel literary translation text):
  - https://huggingface.co/datasets/opus_books

## Scope
- Default mode runs quickly on CPU and validates end-to-end.
- Optional full fine-tuning is available behind a flag.

In [2]:
import json
import time
from difflib import SequenceMatcher

import pandas as pd
from datasets import load_dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    MarianTokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    pipeline,
)

RUN_FULL_FINETUNE = False
LANG_SRC = "en"
LANG_TGT = "fr"
MODEL_ID = "Helsinki-NLP/opus-mt-en-fr"
OUTPUT_DIR = Path("./translation_style_tuner_model")
ARTIFACT_DIR = Path("./artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print("RUN_FULL_FINETUNE:", RUN_FULL_FINETUNE)
print("Model:", MODEL_ID)

RUN_FULL_FINETUNE: False
Model: Helsinki-NLP/opus-mt-en-fr


## Step 1 - Dataset Curation (Domain and Style)

We use `opus_books` and curate for a narrower literary style slice:
- Keep short-to-medium sentence pairs for readable, stable experiments.
- Remove duplicates and missing entries.
- Filter for narrative and dialogue vocabulary to keep a more consistent literary tone.
- Build train/validation/test splits for reproducible comparison.

In [3]:
raw_ds = load_dataset("opus_books", f"{LANG_SRC}-{LANG_TGT}", split="train")
raw_df = raw_ds.to_pandas()

# Flatten translation dict into explicit columns
raw_df["src"] = raw_df["translation"].apply(lambda x: x.get(LANG_SRC, "").strip())
raw_df["tgt"] = raw_df["translation"].apply(lambda x: x.get(LANG_TGT, "").strip())

# Basic quality filtering
df = raw_df[["src", "tgt"]].copy()
df = df[(df["src"] != "") & (df["tgt"] != "")]
df = df.drop_duplicates()

# Sentence length constraints for a stable and fast training demo
df = df[(df["src"].str.len() >= 12) & (df["src"].str.len() <= 220)]
df = df[(df["tgt"].str.len() >= 12) & (df["tgt"].str.len() <= 260)]

# Domain-style curation: emphasize literary narrative and dialogue phrasing
DOMAIN_TERMS = [
    "love", "heart", "sir", "madam", "captain", "night", "house", "room",
    "mother", "father", "friend", "letter", "voice", "dream", "hand", "eyes",
]

term_regex = "|".join(DOMAIN_TERMS)
df_domain = df[df["src"].str.lower().str.contains(term_regex, regex=True)].copy()

# Keep a compact subset for quick execution
subset_size = min(1200, len(df_domain))
df_domain = df_domain.head(subset_size).reset_index(drop=True)

# Split: train/val/test
n_total = len(df_domain)
n_train = int(n_total * 0.80)
n_val = int(n_total * 0.10)

train_df = df_domain.iloc[:n_train].copy()
val_df = df_domain.iloc[n_train:n_train + n_val].copy()
test_df = df_domain.iloc[n_train + n_val:].copy()

print(f"Raw rows: {len(raw_df):,}")
print(f"Curated domain rows: {n_total:,}")
print(f"Train / Val / Test: {len(train_df)} / {len(val_df)} / {len(test_df)}")

curation_report = {
    "dataset": "opus_books",
    "source_url": "https://huggingface.co/datasets/opus_books",
    "lang_pair": f"{LANG_SRC}-{LANG_TGT}",
    "raw_rows": int(len(raw_df)),
    "curated_rows": int(n_total),
    "train_rows": int(len(train_df)),
    "val_rows": int(len(val_df)),
    "test_rows": int(len(test_df)),
    "domain_terms": DOMAIN_TERMS,
}

with open(ARTIFACT_DIR / "curation_report.json", "w", encoding="utf-8") as f:
    json.dump(curation_report, f, indent=2, ensure_ascii=False)

train_df.head(3)

Raw rows: 127,085
Curated domain rows: 1,200
Train / Val / Test: 960 / 120 / 120


,src,tgt
0,"I still say 'our home,' although the house no ...","Je continue à dire « chez nous », bien que la ..."
1,"My father, whom I used to call M. Seurel as di...","Mon père, que j’appelais M. Seurel, comme les ..."
2,My mother taught the infants.,Ma mère faisait la petite classe.


In [4]:
# Baseline translator (pretrained, no domain tuning yet)
baseline_tokenizer = MarianTokenizer.from_pretrained(MODEL_ID)
baseline_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID)

def batch_translate(texts, tokenizer, model):
    encoded = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128,
    )
    generated = model.generate(**encoded, max_new_tokens=128)
    return tokenizer.batch_decode(generated, skip_special_tokens=True)

# Evaluate on a compact test slice for speed
eval_n = min(120, len(test_df))
eval_df = test_df.head(eval_n).copy()

t0 = time.time()
eval_df["baseline_pred"] = batch_translate(eval_df["src"].tolist(), baseline_tokenizer, baseline_model)
baseline_infer_time = time.time() - t0

STYLE_TERMS = [
    "love", "heart", "sir", "madam", "captain", "night", "house", "room",
]

def char_similarity(a, b):
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()

def term_preservation_ratio(src_text, pred_text, terms):
    src_l = src_text.lower()
    pred_l = pred_text.lower()
    present_terms = [t for t in terms if t in src_l]
    if not present_terms:
        return 1.0
    kept = [t for t in present_terms if (t in pred_l)]
    return len(kept) / len(present_terms)

baseline_char_sim = eval_df.apply(lambda r: char_similarity(r["baseline_pred"], r["tgt"]), axis=1).mean()
baseline_term_keep = eval_df.apply(lambda r: term_preservation_ratio(r["src"], r["baseline_pred"], STYLE_TERMS), axis=1).mean()
baseline_len_ratio = (eval_df["baseline_pred"].str.len() / eval_df["tgt"].str.len().clip(lower=1)).mean()

print(f"Baseline char similarity: {baseline_char_sim:.3f}")
print(f"Baseline term preservation: {baseline_term_keep:.3f}")
print(f"Baseline length ratio: {baseline_len_ratio:.3f}")
print(f"Baseline inference on {eval_n} rows: {baseline_infer_time:.2f}s")

Loading weights: 100%|██████████| 258/258 [00:00<00:00, 33475.54it/s]
Both `max_new_tokens` (=128) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Baseline char similarity: 0.509
Baseline term preservation: 0.475
Baseline length ratio: 1.143
Baseline inference on 120 rows: 35.67s


## Step 2 - Fine-Tuning Plan (Domain-Specific Style)

We include a full fine-tuning pipeline with LoRA-free `Seq2SeqTrainer` style tuning.

- **Default**: `RUN_FULL_FINETUNE = False` keeps notebook fast and fully runnable.
- **When enabled**: runs short supervised tuning on curated OPUS Books literary pairs.
- This structure separates experimentation from expensive training, while keeping all code reproducible.

In [5]:
def prepare_hf_dataset(frame):
    from datasets import Dataset
    ds = Dataset.from_pandas(frame[["src", "tgt"]], preserve_index=False)
    return ds

train_ds = prepare_hf_dataset(train_df.head(min(600, len(train_df))))
val_ds = prepare_hf_dataset(val_df.head(min(120, len(val_df))))

tokenizer = MarianTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID)

max_src_len = 96
max_tgt_len = 96

def preprocess(batch):
    model_inputs = tokenizer(batch["src"], max_length=max_src_len, truncation=True)
    labels = tokenizer(text_target=batch["tgt"], max_length=max_tgt_len, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_tok = train_ds.map(preprocess, batched=True, remove_columns=train_ds.column_names)
val_tok = val_ds.map(preprocess, batched=True, remove_columns=val_ds.column_names)

if RUN_FULL_FINETUNE:
    args = Seq2SeqTrainingArguments(
        output_dir=str(OUTPUT_DIR),
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        learning_rate=3e-5,
        num_train_epochs=1,
        logging_steps=20,
        eval_strategy="steps",
        eval_steps=50,
        save_steps=100,
        save_total_limit=1,
        predict_with_generate=True,
        fp16=False,
        bf16=False,
        report_to="none",
        seed=SEED,
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=args,
        train_dataset=train_tok,
        eval_dataset=val_tok,
        data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model),
        tokenizer=tokenizer,
    )

    print("Running full fine-tuning...")
    trainer.train()
    trainer.save_model(str(OUTPUT_DIR))
    tokenizer.save_pretrained(str(OUTPUT_DIR))
    tuned_model_path = str(OUTPUT_DIR)
    print("Saved tuned model to:", tuned_model_path)
else:
    tuned_model_path = MODEL_ID
    print("RUN_FULL_FINETUNE=False -> using pretrained model for this run.")

Map: 100%|██████████| 120/120 [00:00<00:00, 3614.25 examples/s]

RUN_FULL_FINETUNE=False -> using pretrained model for this run.


## Step 3 - Evaluation and Comparison

We evaluate both baseline and tuned model using practical style-aware proxies:
- **Character similarity** against reference translation.
- **Term preservation ratio** for domain lexicon.
- **Length ratio** as a quick adequacy/verbosity check.

These are not perfect quality metrics, but they are transparent and lightweight for iteration.

In [6]:
tuned_tokenizer = MarianTokenizer.from_pretrained(tuned_model_path)
tuned_model = AutoModelForSeq2SeqLM.from_pretrained(tuned_model_path)

t0 = time.time()
eval_df["tuned_pred"] = batch_translate(eval_df["src"].tolist(), tuned_tokenizer, tuned_model)
tuned_infer_time = time.time() - t0

tuned_char_sim = eval_df.apply(lambda r: char_similarity(r["tuned_pred"], r["tgt"]), axis=1).mean()
tuned_term_keep = eval_df.apply(lambda r: term_preservation_ratio(r["src"], r["tuned_pred"], STYLE_TERMS), axis=1).mean()
tuned_len_ratio = (eval_df["tuned_pred"].str.len() / eval_df["tgt"].str.len().clip(lower=1)).mean()

results = pd.DataFrame([
    {
        "approach": "baseline_pretrained",
        "char_similarity": baseline_char_sim,
        "term_preservation": baseline_term_keep,
        "length_ratio": baseline_len_ratio,
        "inference_seconds": baseline_infer_time,
    },
    {
        "approach": "tuned_or_same_model",
        "char_similarity": tuned_char_sim,
        "term_preservation": tuned_term_keep,
        "length_ratio": tuned_len_ratio,
        "inference_seconds": tuned_infer_time,
    },
])

results.to_csv(ARTIFACT_DIR / "evaluation_summary.csv", index=False)
eval_df[["src", "tgt", "baseline_pred", "tuned_pred"]].to_csv(
    ARTIFACT_DIR / "sample_predictions.csv", index=False
)

print(results)
print("Saved:", ARTIFACT_DIR / "evaluation_summary.csv")
print("Saved:", ARTIFACT_DIR / "sample_predictions.csv")

Loading weights: 100%|██████████| 258/258 [00:00<00:00, 28385.98it/s]
Both `max_new_tokens` (=128) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


              approach  char_similarity  term_preservation  length_ratio  \
0  baseline_pretrained         0.508794              0.475      1.143205   
1  tuned_or_same_model         0.508794              0.475      1.143205   

   inference_seconds  
0          35.666412  
1          36.014950  
Saved: artifacts\evaluation_summary.csv
Saved: artifacts\sample_predictions.csv


## Interpretation, Limitations, and Next Steps

### Interpretation
- If `RUN_FULL_FINETUNE=False`, both rows may be similar because both use pretrained weights.
- Once full tuning is enabled, compare changes in:
  - character similarity,
  - term preservation,
  - and qualitative phrase consistency.

### Dataset Curation Notes
- OPUS Books is domain-aligned for literary translation, but still noisy in style consistency.
- Heuristic term filtering improves relevance but can bias lexical distribution.
- Small subsets are good for fast prototyping, not for production-grade claims.

### Limitations
- Metrics here are proxies and do not replace human bilingual review.
- No direct COMET/BLEURT metric in default run.
- Single language pair (`en-fr`) and narrow style terms.
- Fine-tuning quality depends strongly on clean domain pairs and stable hardware.

### Recommended Improvements
1. Add human review rubric (accuracy, fluency, terminology, tone).
2. Expand curation with stricter alignment and noise filtering.
3. Add COMET/BLEU metrics in a richer environment.
4. Tune with longer training and checkpoint selection on validation data.
5. Compare against a stronger translation backbone for the same domain.